# Driver Frame Anonymisation

## Cell 1 — Install Dependencies

# Original for Linus

import subprocess, sys
libs = [
    "insightface",
    "onnxruntime",          # CPU inference backend for InsightFace
    "opencv-python-headless",
]
for lib in libs:
    print(f"Installing {lib}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", lib, "-q"])
    print(f"  ✓ {lib}")
print("\n✓ All dependencies installed — restart kernel if this is first run")

In [ ]:
import subprocess, sys

libs = [
    "mediapipe",
    "opencv-python-headless",
]

for lib in libs:
    print(f"Installing {lib}...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", lib],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"  ✗ FAILED\n{result.stderr}")
    else:
        print(f"  ✓ {lib}")

print("\n✓ Done — restart kernel if first run")

## Cell 2 — Config

In [ ]:
import os

# ── Paths ──────────────────────────────────────────────────────────────────────
SOURCE_BASE_DIR = "debug_frames_qwen"       # original clips — never modified
ANON_BASE_DIR   = "debug_frames_qwen_anon"  # anonymised output
ANON_LOG_PATH   = os.path.join(ANON_BASE_DIR, "anon_log.csv")

os.makedirs(ANON_BASE_DIR, exist_ok=True)

# ── Filenames ──────────────────────────────────────────────────────────────────
RESULT_FILENAME      = "result.json"      # copied as-is (no URLs inside)
SOURCE_META_FILENAME = "source_meta.json" # copied as-is (no URLs inside)
ANON_META_FILENAME   = "anon_meta.json"   # written by this notebook

# ── InsightFace settings ───────────────────────────────────────────────────────
# Detection confidence threshold (0.0–1.0).
# 0.35 is a good balance — catches angled faces without too many false positives.
# InsightFace is much more conservative than MediaPipe so you can go lower safely.
FACE_CONFIDENCE = 0.35

# How much to expand the detected bounding box before blurring.
# 0.25 = 25% padding in each direction so hair/ears/forehead are covered.
FACE_BOX_PADDING = 0.25

# Gaussian blur kernel — must be odd. 51 = strong enough for publication.
BLUR_KERNEL = 51

# JPEG quality for saved anonymised frames.
ANON_JPEG_QUALITY = 90

# InsightFace model — buffalo_sc is the lightweight CPU model (MIT licensed).
# On first run it downloads ~100MB to ~/.insightface/models/
INSIGHTFACE_MODEL = "buffalo_sc"

print("✓ Config loaded")
#print(f"  Source    : {os.path.abspath(SOURCE_BASE_DIR)}")
#print(f"  Output    : {os.path.abspath(ANON_BASE_DIR)}")
print(f"  Model     : {INSIGHTFACE_MODEL}")
print(f"  Confidence: {FACE_CONFIDENCE}")
print(f"  Padding   : {FACE_BOX_PADDING*100:.0f}%")
print(f"  Blur      : {BLUR_KERNEL}x{BLUR_KERNEL} Gaussian")

## Cell 3 — Imports & Helpers

In [ ]:
import json, re, shutil
from pathlib import Path
from typing  import Any, Dict, List, Tuple

import numpy  as np
import cv2
import pandas as pd


def read_json(path: str) -> dict:
    """Safely read a JSON file. Returns {} on any error."""
    if not os.path.exists(path):
        return {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)
        return obj if isinstance(obj, dict) else {}
    except Exception:
        return {}


def write_json(path: str, obj: dict) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def discover_clip_dirs(base_dir: str) -> List[Tuple[int, str]]:
    """
    Scan base_dir for clip_NNN folders.
    Returns sorted list of (clip_no, clip_dir).
    """
    clips = []
    if not os.path.isdir(base_dir):
        print(f"  WARNING: {base_dir} does not exist")
        return clips
    for name in os.listdir(base_dir):
        m = re.fullmatch(r"clip_(\d+)", name)
        if m:
            clips.append((int(m.group(1)), os.path.join(base_dir, name)))
    clips.sort(key=lambda x: x[0])
    return clips


def get_driver_frame_paths(clip_dir: str) -> List[str]:
    """Return sorted list of frame_driver_*.jpg paths in a clip folder."""
    return sorted(str(p) for p in Path(clip_dir).glob("frame_driver_*.jpg"))


print("✓ Imports and helpers loaded")

## Cell 4 — Load InsightFace Model & Build Blur Function

# Original
import insightface
from insightface.app import FaceAnalysis

# Load InsightFace — buffalo_sc is the lightweight CPU-optimised model.
# allowed_modules=["detection"] means we only load the face detector,
# not the recognition/landmark modules — faster and lighter.
print("Loading InsightFace model (downloads on first run)...")
_app = FaceAnalysis(
    name=INSIGHTFACE_MODEL,
    allowed_modules=["detection"],
    providers=["CPUExecutionProvider"],  # CPU only — no GPU required
)
# ctx_id=-1 = CPU. det_size is the input resolution — 320x320 is fast and
# sufficient for dashcam frames. Use (640,640) if you want higher recall.
_app.prepare(ctx_id=-1, det_size=(320, 320), det_thresh=FACE_CONFIDENCE)
print(f"✓ InsightFace ready  model={INSIGHTFACE_MODEL}  det_size=320x320  threshold={FACE_CONFIDENCE}")


def blur_faces_in_image(src_path: str, dst_path: str) -> Dict[str, Any]:
    """
    Load image, detect all faces with InsightFace, apply Gaussian blur
    over each padded bounding box, save to dst_path.

    Returns:
      status : 'ok' or 'error'
      faces  : number of faces blurred
      boxes  : list of {x1,y1,x2,y2,score} dicts
    """
    img_bgr = cv2.imread(src_path)
    if img_bgr is None:
        return {"status": "error", "reason": "unreadable", "faces": 0, "boxes": []}

    h, w   = img_bgr.shape[:2]
    output = img_bgr.copy()
    boxes  = []

    # InsightFace expects BGR — same as OpenCV, no conversion needed
    faces = _app.get(img_bgr)

    for face in faces:
        # bbox is [x1, y1, x2, y2] in pixel coords already
        x1, y1, x2, y2 = face.bbox.astype(int)
        bw = x2 - x1
        bh = y2 - y1

        # Expand box by padding fraction so hair/ears/forehead are covered
        x1 = int(x1 - FACE_BOX_PADDING * bw)
        y1 = int(y1 - FACE_BOX_PADDING * bh)
        x2 = int(x2 + FACE_BOX_PADDING * bw)
        y2 = int(y2 + FACE_BOX_PADDING * bh)

        # Clamp to image bounds
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)

        if x2 <= x1 or y2 <= y1:
            continue

        # Apply Gaussian blur to the face region
        k = BLUR_KERNEL
        output[y1:y2, x1:x2] = cv2.GaussianBlur(output[y1:y2, x1:x2], (k, k), 0)

        boxes.append({
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "score": round(float(face.det_score), 3),
        })

    # Always save — even if no face, we copy the frame so the anon
    # folder is structurally identical to the source
    os.makedirs(os.path.dirname(dst_path), exist_ok=True)
    cv2.imwrite(dst_path, output, [cv2.IMWRITE_JPEG_QUALITY, ANON_JPEG_QUALITY])

    return {"status": "ok", "faces": len(boxes), "boxes": boxes}


print("✓ blur_faces_in_image ready")

In [ ]:
import mediapipe as mp
import cv2
import os
import numpy as np

_BaseOptions        = mp.tasks.BaseOptions
_FaceDetector       = mp.tasks.vision.FaceDetector
_FaceDetectorOptions = mp.tasks.vision.FaceDetectorOptions
_VisionRunningMode  = mp.tasks.vision.RunningMode

_options = _FaceDetectorOptions(
    base_options=_BaseOptions(model_asset_path="blaze_face_short_range.tflite"),
    running_mode=_VisionRunningMode.IMAGE,
    min_detection_confidence=FACE_CONFIDENCE,
)
_detector = _FaceDetector.create_from_options(_options)

print(f"✓ MediaPipe FaceDetection ready  threshold={FACE_CONFIDENCE}")


def blur_faces_in_image(src_path: str, dst_path: str) -> dict:
    img_bgr = cv2.imread(src_path)
    if img_bgr is None:
        return {"status": "error", "reason": "unreadable", "faces": 0, "boxes": []}

    h, w   = img_bgr.shape[:2]
    output = img_bgr.copy()
    boxes  = []

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    results  = _detector.detect(mp_image)

    for det in results.detections:
        bb = det.bounding_box
        x1, y1 = bb.origin_x, bb.origin_y
        bw, bh = bb.width, bb.height
        x2, y2 = x1 + bw, y1 + bh

        # Expand by padding
        x1 = int(x1 - FACE_BOX_PADDING * bw)
        y1 = int(y1 - FACE_BOX_PADDING * bh)
        x2 = int(x2 + FACE_BOX_PADDING * bw)
        y2 = int(y2 + FACE_BOX_PADDING * bh)

        # Clamp
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)

        if x2 <= x1 or y2 <= y1:
            continue

        k = BLUR_KERNEL
        output[y1:y2, x1:x2] = cv2.GaussianBlur(output[y1:y2, x1:x2], (k, k), 0)

        score = det.categories[0].score if det.categories else 0.0
        boxes.append({
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "score": round(float(score), 3),
        })

    os.makedirs(os.path.dirname(dst_path), exist_ok=True)
    cv2.imwrite(dst_path, output, [cv2.IMWRITE_JPEG_QUALITY, ANON_JPEG_QUALITY])

    return {"status": "ok", "faces": len(boxes), "boxes": boxes}


print("✓ blur_faces_in_image ready")

## Cell 5 — Discover Clips & Preview
Shows what is on disk before touching anything.

In [ ]:
all_clips = discover_clip_dirs(SOURCE_BASE_DIR)
print(f"Clip folders found in source : {len(all_clips)}")

# Count how many are already anonymised
already_done = sum(
    1 for clip_no, _ in all_clips
    if read_json(os.path.join(ANON_BASE_DIR, f"clip_{clip_no:03d}", ANON_META_FILENAME)).get("anon_status") == "ok"
)

print(f"Already anonymised           : {already_done}")
print(f"Remaining to process         : {len(all_clips) - already_done}")
print(f"\nOutput: {os.path.abspath(ANON_BASE_DIR)}")
print("\nNote: run Cell 6 with all_clips[:5] first to spot-check before full run.")

## Cell 6 — Run Anonymisation

To test on 5 clips first change `all_clips` to `all_clips[:5]`.
Re-running skips already-done clips automatically.

In [ ]:
# ── Change all_clips[:5] to all_clips for full run ────────────────────────────
clips_to_process = all_clips   # or all_clips[:5] for a test run

print(f"Processing {len(clips_to_process)} clips...")
print(f"Originals in '{SOURCE_BASE_DIR}' are never modified.")
print()

log_rows = []

for idx, (clip_no, src_dir) in enumerate(clips_to_process, start=1):
    anon_dir  = os.path.join(ANON_BASE_DIR, f"clip_{clip_no:03d}")
    meta_path = os.path.join(anon_dir, ANON_META_FILENAME)

    # ── Skip if already done ──────────────────────────────────────────────
    existing = read_json(meta_path)
    if existing.get("anon_status") == "ok":
        log_rows.append(existing)
        if idx % 50 == 0:
            print(f"  [{idx}/{len(clips_to_process)}] skipping already-done...")
        continue

    os.makedirs(anon_dir, exist_ok=True)

    # ── Copy result.json and source_meta.json (no URLs in either) ─────────
    for fname in (RESULT_FILENAME, SOURCE_META_FILENAME):
        src_f = os.path.join(src_dir, fname)
        dst_f = os.path.join(anon_dir, fname)
        if os.path.exists(src_f) and not os.path.exists(dst_f):
            shutil.copy2(src_f, dst_f)

    # ── Process every driver frame ────────────────────────────────────────
    frame_paths   = get_driver_frame_paths(src_dir)
    frame_results = []
    total_faces   = 0
    no_face_count = 0

    for frame_path in frame_paths:
        fname    = os.path.basename(frame_path)
        dst_path = os.path.join(anon_dir, fname)
        res      = blur_faces_in_image(frame_path, dst_path)
        frame_results.append({"file": fname, **res})
        total_faces += res.get("faces", 0)
        if res.get("faces", 0) == 0:
            no_face_count += 1

    # ── Write anon_meta.json ──────────────────────────────────────────────
    clip_meta = {
        "anon_status":         "ok",
        "clip_no":             clip_no,
        "source_hash":         read_json(os.path.join(src_dir, SOURCE_META_FILENAME)).get("source_hash", ""),
        "frames_processed":    len(frame_paths),
        "total_faces_blurred": total_faces,
        "frames_no_face":      no_face_count,
        "blur_kernel":         BLUR_KERNEL,
        "det_threshold":       FACE_CONFIDENCE,
        "model":               INSIGHTFACE_MODEL,
        "frames":              frame_results,
    }
    write_json(meta_path, clip_meta)
    log_rows.append(clip_meta)

    # Progress every 10 clips
    if idx % 10 == 0 or idx == len(clips_to_process):
        print(f"  [{idx}/{len(clips_to_process)}] clip_{clip_no:03d}  "
              f"{len(frame_paths)} frames  "
              f"{total_faces} faces blurred  "
              f"{no_face_count} frames with no face")

# ── Write anon_log.csv ────────────────────────────────────────────────────────
log_df = pd.DataFrame([{
    "clip_no":             r.get("clip_no"),
    "source_hash":         r.get("source_hash"),
    "anon_status":         r.get("anon_status"),
    "frames_processed":    r.get("frames_processed"),
    "total_faces_blurred": r.get("total_faces_blurred"),
    "frames_no_face":      r.get("frames_no_face"),
    "model":               r.get("model"),
} for r in log_rows])

log_df.to_csv(ANON_LOG_PATH, index=False)

ok_df = log_df[log_df["anon_status"] == "ok"]
print(f"\n{'='*60}")
print(f"ANONYMISATION COMPLETE")
print(f"{'='*60}")
print(f"  Model                 : {INSIGHTFACE_MODEL}")
print(f"  Clips processed       : {len(ok_df)}")
print(f"  Total frames          : {ok_df['frames_processed'].sum()}")
print(f"  Total faces blurred   : {ok_df['total_faces_blurred'].sum()}")
print(f"  Frames with no face   : {ok_df['frames_no_face'].sum()}")
print(f"  (No-face frames still copied — driver absent or face not detected)")
print(f"  Log: {os.path.abspath(ANON_LOG_PATH)}")

## Cell 7 — Spot-Check: Original vs Anonymised Side by Side

import matplotlib.pyplot as plt
import matplotlib.image  as mpimg

ok_df      = log_df[log_df["anon_status"] == "ok"]
with_faces = ok_df[ok_df["total_faces_blurred"] > 0]

if len(with_faces) == 0:
    print("No clips with detected faces — run Cell 6 first.")
else:
    sample = with_faces.sample(min(3, len(with_faces)), random_state=7)

    for _, row in sample.iterrows():
        clip_no     = int(row["clip_no"])
        src_dir     = os.path.join(SOURCE_BASE_DIR, f"clip_{clip_no:03d}")
        anon_dir    = os.path.join(ANON_BASE_DIR,   f"clip_{clip_no:03d}")
        src_frames  = get_driver_frame_paths(src_dir)
        anon_frames = get_driver_frame_paths(anon_dir)

        if not src_frames or not anon_frames:
            print(f"  clip_{clip_no:03d}: frames missing")
            continue

        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        axes[0].imshow(mpimg.imread(src_frames[0]))
        axes[0].set_title(f"clip_{clip_no:03d} — ORIGINAL", fontsize=11)
        axes[1].imshow(mpimg.imread(anon_frames[0]))
        axes[1].set_title(f"clip_{clip_no:03d} — ANONYMISED", fontsize=11)
        for ax in axes:
            ax.axis("off")
        plt.tight_layout()
        plt.show()
        print(f"  clip_{clip_no:03d}: {int(row['total_faces_blurred'])} face(s) blurred  "
              f"{int(row['frames_no_face'])} frames with no face")

## Cell 8 — No-Face Audit
Clips where no face was detected across any frame.
These were still copied unchanged — review before publishing.

In [ ]:
ok_df         = log_df[log_df["anon_status"] == "ok"]
no_face_clips = ok_df[ok_df["total_faces_blurred"] == 0]

print(f"Clips with zero faces detected : {len(no_face_clips)} / {len(ok_df)}")
print(f"Clips with faces blurred       : {len(ok_df) - len(no_face_clips)} / {len(ok_df)}")
print()

if len(no_face_clips) > 0:
    print("Clip numbers with no face detected:")
    print(sorted(no_face_clips["clip_no"].astype(int).tolist()))
    print()
    print("Possible reasons:")
    print("  - Driver seat empty")
    print("  - Driver facing away / extreme profile")
    print("  - Night vision / dark frame")
    print("  - Camera-gated clip (covered/blurry)")
    print()
    print("Run Cell 9 to visually inspect these clips.")
else:
    print("✓ All clips had at least one face detected.")

## Cell 9 — Inspect No-Face Clips
Visual check — shows original frames so you can judge if a face was missed.

In [ ]:
import random
import math
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

"""
CELL 9 (UPDATED) — NO-FACE REVIEW
===================================
Shows all no-face frames if the number is manageable,
or a capped sample if there are too many.
Small thumbnails so you can skim quickly.
"""
 
# Max frames to show before switching to a random sample
NO_FACE_DISPLAY_LIMIT = 300
NO_FACE_GRID_COLS     = 6
 
ok_df_nf         = log_df[log_df["anon_status"] == "ok"]
no_face_clips_nf = ok_df_nf[ok_df_nf["total_faces_blurred"] == 0]
 
# Collect all original frame paths from no-face clips
no_face_frame_paths = []
for _, row in no_face_clips_nf.iterrows():
    clip_no   = int(row["clip_no"])
    src_dir   = os.path.join(SOURCE_BASE_DIR, f"clip_{clip_no:03d}")
    paths     = sorted(str(p) for p in Path(src_dir).glob("frame_driver_*.jpg"))
    no_face_frame_paths.extend(paths)
 
total_no_face = len(no_face_frame_paths)
 
print(f"Clips with zero faces         : {len(no_face_clips_nf)} / {len(ok_df_nf)}")
print(f"Total frames in those clips   : {total_no_face}")
 
if total_no_face == 0:
    print("✓ No zero-face clips — nothing to review.")
else:
    if total_no_face <= NO_FACE_DISPLAY_LIMIT:
        display_paths = no_face_frame_paths
        print(f"Showing ALL {total_no_face} frames (under limit of {NO_FACE_DISPLAY_LIMIT})")
    else:
        random.seed(99)
        display_paths = random.sample(no_face_frame_paths, NO_FACE_DISPLAY_LIMIT)
        display_paths.sort()
        print(f"Showing {NO_FACE_DISPLAY_LIMIT} of {total_no_face} frames (random sample — too many to show all)")
 
    print(f"Grid: {math.ceil(len(display_paths)/NO_FACE_GRID_COLS)} rows × {NO_FACE_GRID_COLS} cols")
    print()
 
    n_cols = NO_FACE_GRID_COLS
    n_rows = math.ceil(len(display_paths) / n_cols)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.2, n_rows * 2.0))
    axes_flat = axes.flatten() if hasattr(axes, "flatten") else [axes]
    
    for i, ax in enumerate(axes_flat):
        if i < len(display_paths):
            img = mpimg.imread(display_paths[i])
            ax.imshow(img)
            ax.set_title(Path(display_paths[i]).stem, fontsize=5)
        ax.axis("off")
    
    plt.tight_layout(pad=0.3)
    plt.show()
    print("These are ORIGINAL frames — check if any have a visible face that was missed.")
    if total_no_face > NO_FACE_DISPLAY_LIMIT:
        print(f"NOTE: Only {NO_FACE_DISPLAY_LIMIT} of {total_no_face} shown. "
              f"Increase NO_FACE_DISPLAY_LIMIT to see more.")

# Do a spot check on blurred images

In [ ]:
"""
CELL 10 — BULK SPOT-CHECK: BLURRED FRAMES
==========================================
Shows a large sample of anonymised frames as a grid so you can
quickly skim for missed faces or over-blurring.

Also paste the updated Cell 9 below to replace your existing one.
"""

# ─────────────────────────────────────────────────────────────────────────────
# CELL 10A — COUNT FIRST (run this before displaying anything)
# ─────────────────────────────────────────────────────────────────────────────

import os
import random
import math
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.image  as mpimg
import pandas as pd

# How many frames to show per row in the grid
GRID_COLS = 6

# Max frames to display in one go — adjust if your machine is slow
MAX_DISPLAY = 300

# Sample size as fraction of available frames — we pick this many for the grid
SAMPLE_FRACTION = 0.15   # 15% of all blurred frames

ok_df      = log_df[log_df["anon_status"] == "ok"]
with_faces = ok_df[ok_df["total_faces_blurred"] > 0]

# Collect all anonymised driver frame paths from clips that had faces blurred
all_blurred_frame_paths = []
for _, row in with_faces.iterrows():
    clip_no  = int(row["clip_no"])
    anon_dir = os.path.join(ANON_BASE_DIR, f"clip_{clip_no:03d}")
    paths    = sorted(str(p) for p in Path(anon_dir).glob("frame_driver_*.jpg"))
    all_blurred_frame_paths.extend(paths)

total_available = len(all_blurred_frame_paths)
sample_n        = min(MAX_DISPLAY, max(1, int(total_available * SAMPLE_FRACTION)))

print(f"Clips with faces blurred     : {len(with_faces)}")
print(f"Total anonymised frames      : {total_available}")
print(f"Sample fraction              : {SAMPLE_FRACTION*100:.0f}%")
print(f"Frames that will be shown    : {sample_n}")
print(f"Grid size                    : {sample_n // GRID_COLS + 1} rows × {GRID_COLS} cols")
print()
print("If happy with the count, run Cell 10b to display the grid.")
print("Adjust MAX_DISPLAY or SAMPLE_FRACTION at the top of this cell if needed.")

In [ ]:
"""
CELL 10B — DISPLAY BLURRED FRAME GRID
======================================
Run Cell 10A first to confirm the count, then run this.
"""
 
# Sample randomly with fixed seed for reproducibility
random.seed(42)
sample_paths = random.sample(all_blurred_frame_paths, sample_n)
sample_paths.sort()  # sort so clips are roughly in order for easier review
 
n_cols = GRID_COLS
n_rows = math.ceil(len(sample_paths) / n_cols)
 
# Small thumbnail size — just enough to spot a missed face quickly
fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(n_cols * 2.2, n_rows * 2.0),  # 2.2 x 2.0 inches per thumbnail
)
 
# Flatten axes so we can index linearly
axes_flat = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes.flatten()
 
for i, ax in enumerate(axes_flat):
    if i < len(sample_paths):
        try:
            img = mpimg.imread(sample_paths[i])
            ax.imshow(img)
            # Show clip number as tiny label so you know which clip to investigate
            clip_label = Path(sample_paths[i]).parent.name  # e.g. clip_042
            frame_label = Path(sample_paths[i]).stem         # e.g. frame_driver_03
            ax.set_title(f"{clip_label}\n{frame_label}", fontsize=5, pad=1)
        except Exception:
            ax.set_facecolor("#222")
    ax.axis("off")
 
plt.suptitle(
    f"Anonymised frame spot-check — {len(sample_paths)} frames "
    f"({SAMPLE_FRACTION*100:.0f}% sample of {total_available} total)",
    fontsize=9, y=1.01,
)
plt.tight_layout(pad=0.3)
plt.show()
print(f"Displayed {len(sample_paths)} frames. Any concerns? Note the clip_NNN label and inspect that folder.")
 

# Clean up folder:

In [ ]:
import json
import os
import re
import shutil
from collections import Counter

ANON_CLEAN_DIR = ANON_BASE_DIR.rstrip("/") + "_clean"

VLM_CODE_TO_EVENT_DESCRIPTION = {
    4:  "v_cam_covered",
    5:  "SEATBELT_D_OFF",
    6:  "v_cam_covered",
    10: "v_phone",
    11: "v_distraction",
    13: "v_fatigue",
    15: "v_smoke",
    66: "DRIVER_FACE_OBSTRUCTED",
    67: "v_cam_covered_review",   # tier 2 — possible covering
}

def _get_combo_label(result: dict) -> str:
    codes = set()
    for item in result.get("det", []) or []:
        try:
            codes.add(int(item.get("e")))
        except Exception:
            continue
    labels = set()
    for c in codes:
        desc = VLM_CODE_TO_EVENT_DESCRIPTION.get(c)
        if desc:
            labels.add(desc)
    if not labels:
        return "no_detection"
    return " + ".join(sorted(labels))

# Fresh destination
if os.path.exists(ANON_CLEAN_DIR):
    print(f"Removing existing: {ANON_CLEAN_DIR}")
    shutil.rmtree(ANON_CLEAN_DIR)
os.makedirs(ANON_CLEAN_DIR, exist_ok=True)

copied      = 0
no_result   = 0
label_count = Counter()

clip_dirs = sorted(
    name for name in os.listdir(ANON_BASE_DIR)
    if re.fullmatch(r"clip_\d+", name)
)

print(f"Source      : {ANON_BASE_DIR}")
print(f"Destination : {ANON_CLEAN_DIR}")
print(f"Clips found : {len(clip_dirs)}")
print("Copying...")

for name in clip_dirs:
    src_clip    = os.path.join(ANON_BASE_DIR, name)
    result_path = os.path.join(src_clip, "result.json")

    if not os.path.exists(result_path):
        label = "unprocessed"
        no_result += 1
    else:
        try:
            with open(result_path) as f:
                result = json.load(f)
            label = _get_combo_label(result)
        except Exception:
            label = "unprocessed"
            no_result += 1

    dst_clip = os.path.join(ANON_CLEAN_DIR, label, name)
    os.makedirs(os.path.join(ANON_CLEAN_DIR, label), exist_ok=True)
    shutil.copytree(src_clip, dst_clip)
    label_count[label] += 1
    copied += 1

print(f"\nDone.")
print(f"  Copied      : {copied}")
print(f"  Unprocessed : {no_result}")
print(f"\nSubfolders created (sorted by clip count):")
print(f"  {'Label':<55} {'Clips':>6}")
print(f"  {'─'*63}")
for label, count in label_count.most_common():
    print(f"  {label:<55} {count:>6}")